In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: NVIDIA H200 NVL


# Code Evaluation: Filter Heads Circuit Analysis

## Overview

This evaluation notebook provides a **static code inspection** of the filter heads circuit analysis codebase at `/net/scratch2/smallyan/filter_eval`. The evaluation is performed according to the plan.md and CodeWalkthrough.md specifications.

### Project Goal (from plan.md)
Investigate the mechanisms underlying list-processing tasks in LLMs to understand how they encode and execute filtering operations. The core hypothesis is that a small number of attention heads ("filter heads") encode a compact representation of the filtering predicate in their query states.

### Main Code Components (from CodeWalkthrough.md)
1. **demo.ipynb** - Main demo notebook showing filter head behavior
2. **scripts/locate_selection_heads.py** - Script to locate filter heads using DCM
3. **src/selection/data.py** - Data handling for selection tasks
4. **src/selection/functional.py** - Core functions for filter head analysis
5. **src/selection/optimization.py** - Optimization for finding optimal head masks
6. **src/models.py** - Model loading and configuration
7. **src/functional.py** - Core functional utilities

## Evaluation Criteria
- **Runnable (Y/N)**: Syntactically valid, required imports/definitions available in scope, no obvious runtime blockers
- **Correct-Implementation (Y/N)**: Logic implements the described computation correctly per Plan/codewalk intent
- **Redundant (Y/N)**: Duplicates another block's computation without adding new information
- **Irrelevant (Y/N)**: Does not contribute to achieving the project goal

In [2]:
# Code Evaluation - Static Analysis
import json
import pandas as pd

# Define the complete evaluation results
# Evaluating all major code files/notebooks for:
# - Runnable (Y/N): Syntactically valid, imports available, no obvious runtime blockers
# - Correct-Implementation (Y/N): Logic implements described computation correctly
# - Redundant (Y/N): Duplicates another block's computation
# - Irrelevant (Y/N): Does not contribute to project goal

evaluation_results = [
    # ==========================================
    # FILE 1: demo.ipynb - Main Demo Notebook
    # ==========================================
    {"File": "demo.ipynb", "Block/Function": "Cell 1: autoreload", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Standard IPython autoreload setup for development."},
    {"File": "demo.ipynb", "Block/Function": "Cell 2: imports & model loading", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Loads ModelandTokenizer with Llama-3.3-70B-Instruct, bfloat16, eager attention."},
    {"File": "demo.ipynb", "Block/Function": "Cell 3: filter head selection", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Selects layer_idx=35, head_idx=19 per plan specification."},
    {"File": "demo.ipynb", "Block/Function": "Cell 4: markdown header", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Documentation markdown cell."},
    {"File": "demo.ipynb", "Block/Function": "Cell 5: SelectOneTask setup", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Loads SelectOneTask from data_save/selection/objects.json."},
    {"File": "demo.ipynb", "Block/Function": "Cell 6: get_random_sample", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Gets random sample with filter_by_lm_prediction=True for fruit category."},
    {"File": "demo.ipynb", "Block/Function": "Cell 7: verify_head_patterns", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Visualizes attention pattern of selected filter head."},
    {"File": "demo.ipynb", "Block/Function": "Cell 8: markdown patching", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Documentation explaining query state patching."},
    {"File": "demo.ipynb", "Block/Function": "Cell 9: counterfactual samples", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Creates source/destination sample pair for patching."},
    {"File": "demo.ipynb", "Block/Function": "Cell 10: Figure 1 setup", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Manual setup to replicate Figure 1 from paper."},
    {"File": "demo.ipynb", "Block/Function": "Cell 11: attention analysis", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Runs verify_head_patterns on source and destination."},
    {"File": "demo.ipynb", "Block/Function": "Cell 12: shape check", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Debug cell checking logits shape."},
    {"File": "demo.ipynb", "Block/Function": "Cell 13: single head patching", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Core patching logic - caches q projections, patches single head."},
    {"File": "demo.ipynb", "Block/Function": "Cell 14: markdown all heads", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Documentation cell."},
    {"File": "demo.ipynb", "Block/Function": "Cell 15: filter_heads dict", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Filter heads for Llama-3.3-70B and gemma-2-27b."},
    {"File": "demo.ipynb", "Block/Function": "Cell 16: all heads attention", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Attention patterns for all filter heads."},
    {"File": "demo.ipynb", "Block/Function": "Cell 17: all heads patching", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Patches all 79 filter heads, shows delta score improvement."},
    {"File": "demo.ipynb", "Block/Function": "Cells 18-19: empty", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "Y", "Irrelevant": "Y", "Notes": "Empty cells at end - should be removed."},
]

print(f"demo.ipynb entries: {len([e for e in evaluation_results if e['File'] == 'demo.ipynb'])}")

demo.ipynb entries: 18


In [3]:
# Continue adding evaluations

# ==========================================
# FILE 2: scripts/locate_selection_heads.py
# ==========================================
evaluation_results.extend([
    {"File": "scripts/locate_selection_heads.py", "Block/Function": "prepare_dataset()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Prepares train/validation datasets with counterfactual samples."},
    {"File": "scripts/locate_selection_heads.py", "Block/Function": "validate()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Validates selected heads using q_proj intervention."},
    {"File": "scripts/locate_selection_heads.py", "Block/Function": "load_dataset()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Loads pre-saved dataset from JSON files."},
    {"File": "scripts/locate_selection_heads.py", "Block/Function": "find_optimal_masks()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Wrapper for optimization and validation."},
    {"File": "scripts/locate_selection_heads.py", "Block/Function": "__main__ block", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Entry point: parses args, loads model, runs optimization."},
])

# ==========================================
# FILE 3: src/selection/data.py
# ==========================================
evaluation_results.extend([
    {"File": "src/selection/data.py", "Block/Function": "SelectionSample dataclass", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Core dataclass for selection samples with prompt templates."},
    {"File": "src/selection/data.py", "Block/Function": "MCQify_sample()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Converts samples to MCQ format."},
    {"File": "src/selection/data.py", "Block/Function": "SelectAllSample dataclass", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Sample class for selecting all items."},
    {"File": "src/selection/data.py", "Block/Function": "CountingSample dataclass", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Sample class for counting task."},
    {"File": "src/selection/data.py", "Block/Function": "YesNoSample dataclass", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Sample class for yes/no presence checking."},
    {"File": "src/selection/data.py", "Block/Function": "DeductionSample dataclass", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Sample class for deduction tasks."},
    {"File": "src/selection/data.py", "Block/Function": "CounterFactualSamplePair", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Pairs clean and patch samples for CF experiments."},
    {"File": "src/selection/data.py", "Block/Function": "SelectOneTask dataclass", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Task class managing categories and templates."},
    {"File": "src/selection/data.py", "Block/Function": "Other Task classes", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "CountingTask, YesNoTask, SelectFirst/Last follow same pattern."},
    {"File": "src/selection/data.py", "Block/Function": "get_counterfactual_* functions", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Functions to generate CF sample pairs for each task."},
])

# ==========================================
# FILE 4: src/selection/functional.py
# ==========================================
evaluation_results.extend([
    {"File": "src/selection/functional.py", "Block/Function": "get_patches_to_verify_independent_enrichment()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Creates patches for independent enrichment verification."},
    {"File": "src/selection/functional.py", "Block/Function": "verify_head_patterns()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Core function for attention visualization with patches."},
    {"File": "src/selection/functional.py", "Block/Function": "cache_q_projections()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Caches Q projections using nnsight, handles GQA."},
    {"File": "src/selection/functional.py", "Block/Function": "find_quesmark_pos()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Finds question mark position for patching."},
])

print(f"Total entries so far: {len(evaluation_results)}")

Total entries so far: 37


In [4]:
# Continue adding evaluations

# ==========================================
# FILE 5: src/selection/optimization.py
# ==========================================
evaluation_results.extend([
    {"File": "src/selection/optimization.py", "Block/Function": "get_optimal_head_mask_optimized()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Optimized DCM. Learns binary mask over heads with L1 sparsity."},
    {"File": "src/selection/optimization.py", "Block/Function": "get_optimal_head_mask_prev()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Legacy optimization, faster for question-after format."},
    {"File": "src/selection/optimization.py", "Block/Function": "validate_q_proj_ie_on_sample_pair()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Validates intervention on single sample pair."},
    {"File": "src/selection/optimization.py", "Block/Function": "Commented get_optimal_head_mask()", "Runnable": "N", "Correct-Implementation": "Y", "Redundant": "Y", "Irrelevant": "Y", "Notes": "Commented-out code (lines 46-397). Should be removed."},
])

# ==========================================
# FILE 6: src/models.py
# ==========================================
evaluation_results.extend([
    {"File": "src/models.py", "Block/Function": "ModelandTokenizer class", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Extends nnsight LanguageModel. Supports multiple model families."},
    {"File": "src/models.py", "Block/Function": "parse_config()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Extracts n_layer, n_embd, layer name formats."},
    {"File": "src/models.py", "Block/Function": "cache_forwards/reset_forward", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Caches and resets forward passes for interventions."},
    {"File": "src/models.py", "Block/Function": "LMHead class", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Combines final layer norm and lm_head."},
    {"File": "src/models.py", "Block/Function": "is_*_variant() helpers", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Model family detection functions."},
    {"File": "src/models.py", "Block/Function": "determine_*_path() helpers", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Functions to determine layer paths."},
])

# ==========================================
# FILE 7: src/functional.py
# ==========================================
evaluation_results.extend([
    {"File": "src/functional.py", "Block/Function": "interpret_logits()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Converts logits to PredictedToken with prob/rank tracking."},
    {"File": "src/functional.py", "Block/Function": "logit_lens()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Projects hidden states to vocabulary."},
    {"File": "src/functional.py", "Block/Function": "patchscope()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Patchscope technique for probing hidden states."},
    {"File": "src/functional.py", "Block/Function": "PatchSpec dataclass", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Specifies location, patch, strategy for interventions."},
    {"File": "src/functional.py", "Block/Function": "generate_with_patch()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Generates text with patching during forward pass."},
    {"File": "src/functional.py", "Block/Function": "predict_next_token()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Predicts next token(s) with optional patching."},
    {"File": "src/functional.py", "Block/Function": "get_module_nnsight()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Retrieves module by dotted path name."},
    {"File": "src/functional.py", "Block/Function": "patch_with_baukit()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Patches using baukit TraceDict."},
    {"File": "src/functional.py", "Block/Function": "patch_with_nnsight()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Alternative patching using nnsight. Has debug prints."},
    {"File": "src/functional.py", "Block/Function": "get_hs()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Gets hidden states at specified locations."},
    {"File": "src/functional.py", "Block/Function": "patch_linear_subspaces()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Patches in rotated subspace for DAS-style."},
    {"File": "src/functional.py", "Block/Function": "free_gpu_cache()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Clears GPU cache with synchronization."},
    {"File": "src/functional.py", "Block/Function": "repeat_kv()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Repeats KV heads for grouped query attention."},
    {"File": "src/functional.py", "Block/Function": "Commented obsolete code", "Runnable": "N", "Correct-Implementation": "Y", "Redundant": "Y", "Irrelevant": "Y", "Notes": "Lines 1104-1199 contain commented-out Bridge code."},
])

print(f"Total entries so far: {len(evaluation_results)}")

Total entries so far: 61


In [5]:
# Continue adding evaluations for supporting files

# ==========================================
# FILE 8: src/attention.py
# ==========================================
evaluation_results.extend([
    {"File": "src/attention.py", "Block/Function": "AttentionInformation dataclass", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Stores tokenized prompt and attention matrices."},
    {"File": "src/attention.py", "Block/Function": "repeat_kv()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "Y", "Irrelevant": "N", "Notes": "Duplicates repeat_kv from src/functional.py."},
    {"File": "src/attention.py", "Block/Function": "get_attention_matrices()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Extracts attention matrices with optional value weighting."},
    {"File": "src/attention.py", "Block/Function": "visualize_attn_matrix()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Visualizes attention using circuitsvis."},
    {"File": "src/attention.py", "Block/Function": "visualize_average_attn_matrix()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Visualizes average attention across heads."},
])

# ==========================================
# FILE 9: src/tokens.py
# ==========================================
evaluation_results.extend([
    {"File": "src/tokens.py", "Block/Function": "prepare_offset_mapping()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Workaround for LLaMA3 tokenizer bug."},
    {"File": "src/tokens.py", "Block/Function": "prepare_input()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Prepares input for model with proper padding."},
    {"File": "src/tokens.py", "Block/Function": "find_token_range()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Finds token indices for substring in string."},
    {"File": "src/tokens.py", "Block/Function": "insert_padding_before_pos()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Inserts padding tokens for alignment."},
    {"File": "src/tokens.py", "Block/Function": "align_patching_positions()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Aligns positions between clean and patched inputs."},
    {"File": "src/tokens.py", "Block/Function": "align_bridge_entities_in_query()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Aligns bridge entities (may be obsolete with Bridge dataset)."},
])

# ==========================================
# FILE 10: src/selection/utils.py
# ==========================================
evaluation_results.extend([
    {"File": "src/selection/utils.py", "Block/Function": "get_first_token_id()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Gets first token ID for a name."},
    {"File": "src/selection/utils.py", "Block/Function": "KeyedSet class", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Set with token ID keys for option tracking."},
    {"File": "src/selection/utils.py", "Block/Function": "verify_correct_option()", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Verifies if model selects correct option."},
])

# ==========================================
# FILE 11: src/utils/env_utils.py
# ==========================================
evaluation_results.extend([
    {"File": "src/utils/env_utils.py", "Block/Function": "env.yml loading", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Loads config from env.yml with fallback defaults."},
    {"File": "src/utils/env_utils.py", "Block/Function": "determine_*_dir() functions", "Runnable": "Y", "Correct-Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Functions to determine data/model/results directories."},
])

print(f"Total entries: {len(evaluation_results)}")

Total entries: 77


## Block-Level Evaluation Table

In [6]:
# Create DataFrame and display the evaluation table
df = pd.DataFrame(evaluation_results)

# Display the full table
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', None)
print("=" * 120)
print("BLOCK-LEVEL EVALUATION TABLE")
print("=" * 120)
print(df.to_string(index=False))
print("=" * 120)

BLOCK-LEVEL EVALUATION TABLE
                             File                                 Block/Function Runnable Correct-Implementation Redundant Irrelevant                                                                           Notes
                       demo.ipynb                             Cell 1: autoreload        Y                      Y         N          N                              Standard IPython autoreload setup for development.
                       demo.ipynb                Cell 2: imports & model loading        Y                      Y         N          N Loads ModelandTokenizer with Llama-3.3-70B-Instruct, bfloat16, eager attention.
                       demo.ipynb                  Cell 3: filter head selection        Y                      Y         N          N                       Selects layer_idx=35, head_idx=19 per plan specification.
                       demo.ipynb                        Cell 4: markdown header        Y                      Y   

## Quantitative Metrics

In [7]:
# Calculate quantitative metrics
total_blocks = len(df)

# Count each flag
runnable_y = (df['Runnable'] == 'Y').sum()
runnable_n = (df['Runnable'] == 'N').sum()

correct_y = (df['Correct-Implementation'] == 'Y').sum()
correct_n = (df['Correct-Implementation'] == 'N').sum()

redundant_y = (df['Redundant'] == 'Y').sum()
redundant_n = (df['Redundant'] == 'N').sum()

irrelevant_y = (df['Irrelevant'] == 'Y').sum()
irrelevant_n = (df['Irrelevant'] == 'N').sum()

# Calculate percentages
runnable_pct = (runnable_y / total_blocks) * 100
incorrect_pct = (correct_n / total_blocks) * 100
redundant_pct = (redundant_y / total_blocks) * 100
irrelevant_pct = (irrelevant_y / total_blocks) * 100

# Correction rate: blocks where we identified potential fixes for Runnable=N or Correct-Implementation=N
# Based on inspection, the issues are:
# 1. Commented-out code (2 blocks) - fix is to remove them
# 2. Empty cells (1 block) - fix is to remove them
# All identified issues have clear static fixes

failed_blocks = runnable_n + correct_n  # = 2 (both are the same blocks with commented code)
# Note: The 2 blocks with Runnable=N are commented-out code sections that can simply be removed
# The redundant/irrelevant empty cells don't fail Runnable, they just should be cleaned up

# Correction rate for blocks that failed Runnable or Correct-Implementation
# Both failing blocks (commented code) have identified fixes: simply remove them
corrections_identified = 2  # The 2 commented-out code blocks can be fixed by removal
correction_rate_pct = (corrections_identified / max(1, runnable_n)) * 100 if runnable_n > 0 else 0.0

print("=" * 80)
print("QUANTITATIVE METRICS")
print("=" * 80)
print(f"\nTotal code blocks evaluated: {total_blocks}")
print()
print(f"Runnable%:              {runnable_pct:.2f}% ({runnable_y}/{total_blocks})")
print(f"Incorrect%:             {incorrect_pct:.2f}% ({correct_n}/{total_blocks})")
print(f"Redundant%:             {redundant_pct:.2f}% ({redundant_y}/{total_blocks})")
print(f"Irrelevant%:            {irrelevant_pct:.2f}% ({irrelevant_y}/{total_blocks})")
print(f"Correction-Rate%:       {correction_rate_pct:.2f}% ({corrections_identified}/{runnable_n} blocks with identified fixes)")
print("=" * 80)

# Store metrics for JSON output
metrics = {
    "Runnable_Percentage": round(runnable_pct, 2),
    "Incorrect_Percentage": round(incorrect_pct, 2),
    "Redundant_Percentage": round(redundant_pct, 2),
    "Irrelevant_Percentage": round(irrelevant_pct, 2),
    "Correction_Rate_Percentage": round(correction_rate_pct, 2)
}

QUANTITATIVE METRICS

Total code blocks evaluated: 77

Runnable%:              97.40% (75/77)
Incorrect%:             0.00% (0/77)
Redundant%:             5.19% (4/77)
Irrelevant%:            3.90% (3/77)
Correction-Rate%:       100.00% (2/2 blocks with identified fixes)


## Binary Checklist Summary

In [8]:
# Binary Checklist Summary

# C1: All core analysis code is runnable
c1_pass = runnable_n == 0
c1_status = "PASS" if c1_pass else "FAIL"
c1_rationale = "All active code blocks are syntactically valid and runnable." if c1_pass else f"{runnable_n} block(s) have Runnable=N (commented-out code that should be removed)."

# C2: All implementations are correct
c2_pass = correct_n == 0
c2_status = "PASS" if c2_pass else "FAIL"
c2_rationale = "All implementations correctly follow the Plan/codewalk specifications." if c2_pass else f"{correct_n} block(s) have implementation errors."

# C3: No redundant code
c3_pass = redundant_y == 0
c3_status = "PASS" if c3_pass else "FAIL"
c3_rationale = "No redundant code found." if c3_pass else f"{redundant_y} block(s) are redundant: (1) commented-out old optimization code in optimization.py, (2) commented-out Bridge dataset code in functional.py, (3) duplicate repeat_kv() in attention.py, (4) empty cells in demo.ipynb."

# C4: No irrelevant code
c4_pass = irrelevant_y == 0
c4_status = "PASS" if c4_pass else "FAIL"
c4_rationale = "All code contributes to the project goal." if c4_pass else f"{irrelevant_y} block(s) are irrelevant: commented-out obsolete code and empty notebook cells."

# Create checklist table
checklist = [
    {"Checklist Item": "C1", "Condition": "All core analysis code is runnable", "PASS/FAIL": c1_status},
    {"Checklist Item": "C2", "Condition": "All implementations are correct", "PASS/FAIL": c2_status},
    {"Checklist Item": "C3", "Condition": "No redundant code", "PASS/FAIL": c3_status},
    {"Checklist Item": "C4", "Condition": "No irrelevant code", "PASS/FAIL": c4_status},
]

checklist_df = pd.DataFrame(checklist)

print("=" * 80)
print("BINARY CHECKLIST SUMMARY")
print("=" * 80)
print()
print(checklist_df.to_string(index=False))
print()
print("=" * 80)
print("\nRationales:")
print(f"  C1: {c1_rationale}")
print(f"  C2: {c2_rationale}")
print(f"  C3: {c3_rationale}")
print(f"  C4: {c4_rationale}")
print("=" * 80)

BINARY CHECKLIST SUMMARY

Checklist Item                          Condition PASS/FAIL
            C1 All core analysis code is runnable      FAIL
            C2    All implementations are correct      PASS
            C3                  No redundant code      FAIL
            C4                 No irrelevant code      FAIL


Rationales:
  C1: 2 block(s) have Runnable=N (commented-out code that should be removed).
  C2: All implementations correctly follow the Plan/codewalk specifications.
  C3: 4 block(s) are redundant: (1) commented-out old optimization code in optimization.py, (2) commented-out Bridge dataset code in functional.py, (3) duplicate repeat_kv() in attention.py, (4) empty cells in demo.ipynb.
  C4: 3 block(s) are irrelevant: commented-out obsolete code and empty notebook cells.


## Summary

### Overall Assessment

The filter heads circuit analysis codebase is **well-implemented and functionally correct**. The core analysis implements the methodology described in the plan.md:

1. **Causal mediation analysis** using activation patching on list-processing tasks
2. **DCM optimization** to learn sparse binary masks over attention heads  
3. **Query state patching** to transfer predicates between contexts
4. **Multi-task support** for SelectOne, SelectFirst, SelectLast, Counting, and CheckPresence tasks

### Key Findings

**Strengths:**
- All active code is syntactically valid and implements the described algorithms correctly
- Clear separation between data handling, functional utilities, and optimization
- Supports multiple model architectures (Llama, Gemma, Qwen, OLMo, GPT variants)
- Comprehensive demo notebook demonstrating the core filter head patching mechanism

**Issues Identified (all minor and easily fixable):**
1. **Commented-out obsolete code** in `src/selection/optimization.py` (lines 46-397) and `src/functional.py` (lines 1104-1199) - should be removed
2. **Duplicate `repeat_kv()` function** in `src/attention.py` that already exists in `src/functional.py`
3. **Empty cells** at the end of `demo.ipynb` - should be removed
4. **Debug print statements** in `patch_with_nnsight()` - should be removed for cleaner output

### Recommendation

The codebase is ready for use. For cleaner maintenance, remove the commented-out obsolete code and duplicate functions identified above.

In [9]:
# Save JSON summary
json_summary = {
    "Runnable_Percentage": round(runnable_pct, 2),
    "Incorrect_Percentage": round(incorrect_pct, 2),
    "Redundant_Percentage": round(redundant_pct, 2),
    "Irrelevant_Percentage": round(irrelevant_pct, 2),
    "Correction_Rate_Percentage": round(correction_rate_pct, 2),
    
    "Issues": {
        "Runnable_Issues_Exist": runnable_n > 0,
        "Output_Mismatch_Exists": False,  # Not applicable in no-execution mode
        "Incorrect_Exists": correct_n > 0,
        "Redundant_Exists": redundant_y > 0,
        "Irrelevant_Exists": irrelevant_y > 0
    },
    
    "Checklist": {
        "C1_All_Runnable": c1_status,
        "C2_All_Correct": c2_status,
        "C3_No_Redundant": c3_status,
        "C4_No_Irrelevant": c4_status
    },
    
    "Rationale": {
        "C1_All_Runnable": c1_rationale,
        "C2_All_Correct": c2_rationale,
        "C3_No_Redundant": c3_rationale,
        "C4_No_Irrelevant": c4_rationale
    }
}

# Save to JSON file
json_path = "/net/scratch2/smallyan/filter_eval/no_exe_evaluation/code_critic_summary.json"
with open(json_path, 'w') as f:
    json.dump(json_summary, f, indent=2)

print(f"JSON summary saved to: {json_path}")
print("\nJSON content:")
print(json.dumps(json_summary, indent=2))

TypeError: Object of type bool is not JSON serializable

In [10]:
# Fix the numpy bool issue - convert to Python bool
import numpy as np

# Save JSON summary with proper type conversion
json_summary = {
    "Runnable_Percentage": float(round(runnable_pct, 2)),
    "Incorrect_Percentage": float(round(incorrect_pct, 2)),
    "Redundant_Percentage": float(round(redundant_pct, 2)),
    "Irrelevant_Percentage": float(round(irrelevant_pct, 2)),
    "Correction_Rate_Percentage": float(round(correction_rate_pct, 2)),
    
    "Issues": {
        "Runnable_Issues_Exist": bool(runnable_n > 0),
        "Output_Mismatch_Exists": False,  # Not applicable in no-execution mode
        "Incorrect_Exists": bool(correct_n > 0),
        "Redundant_Exists": bool(redundant_y > 0),
        "Irrelevant_Exists": bool(irrelevant_y > 0)
    },
    
    "Checklist": {
        "C1_All_Runnable": c1_status,
        "C2_All_Correct": c2_status,
        "C3_No_Redundant": c3_status,
        "C4_No_Irrelevant": c4_status
    },
    
    "Rationale": {
        "C1_All_Runnable": c1_rationale,
        "C2_All_Correct": c2_rationale,
        "C3_No_Redundant": c3_rationale,
        "C4_No_Irrelevant": c4_rationale
    }
}

# Save to JSON file
json_path = "/net/scratch2/smallyan/filter_eval/no_exe_evaluation/code_critic_summary.json"
with open(json_path, 'w') as f:
    json.dump(json_summary, f, indent=2)

print(f"JSON summary saved to: {json_path}")
print("\nJSON content:")
print(json.dumps(json_summary, indent=2))

JSON summary saved to: /net/scratch2/smallyan/filter_eval/no_exe_evaluation/code_critic_summary.json

JSON content:
{
  "Runnable_Percentage": 97.4,
  "Incorrect_Percentage": 0.0,
  "Redundant_Percentage": 5.19,
  "Irrelevant_Percentage": 3.9,
  "Correction_Rate_Percentage": 100.0,
  "Issues": {
    "Runnable_Issues_Exist": true,
    "Output_Mismatch_Exists": false,
    "Incorrect_Exists": false,
    "Redundant_Exists": true,
    "Irrelevant_Exists": true
  },
  "Checklist": {
    "C1_All_Runnable": "FAIL",
    "C2_All_Correct": "PASS",
    "C3_No_Redundant": "FAIL",
    "C4_No_Irrelevant": "FAIL"
  },
  "Rationale": {
    "C1_All_Runnable": "2 block(s) have Runnable=N (commented-out code that should be removed).",
    "C2_All_Correct": "All implementations correctly follow the Plan/codewalk specifications.",
    "C3_No_Redundant": "4 block(s) are redundant: (1) commented-out old optimization code in optimization.py, (2) commented-out Bridge dataset code in functional.py, (3) duplicate